# Speech Commands Keyword Spotting

<p align="right">
Run Time: ~20 minutes with training included / 1 minute with training skipped
</p>

This notebook walks through the complete pipeline to train, quantize, convert, and
benchmark a DS-CNN model on the **Speech Commands** dataset for Akida 1 hardware.

The Speech Commands dataset has 10 classes (yes, no, up, down, left, right, on, off, stop, go) plus silence and unknown.

The pipeline follows the standard Akida workflow:
1. Train a float model
2. Post-training quantization (PTQ)
3. Quantization-aware training (QAT) fine-tuning
4. Conversion to Akida `.fbz` format
5. Hardware evaluation and benchmarking

In [14]:
import os
import numpy as np
import tensorflow as tf

from tf_keras.utils import set_random_seed

from cnn2snn import load_quantized_model

DATA_PATH = './data/sc10'
CONFIG_PATH = './configs/training_cfg.yml'
MODELS_DIR = './models_notebook/'
os.makedirs(MODELS_DIR, exist_ok=True)

RUN_FLOAT_TRAINING = True
RUN_QAT_TRAINING = True

# Must be called before any TF ops to make GPU ops (conv backward passes,
# bilinear resize, etc.) deterministic. Has a small throughput cost.
tf.config.experimental.enable_op_determinism()

## Dataset

The **Speech Commands** dataset is loaded via TensorFlow Datasets (`speech_commands`).
On the first run, TFDS will automatically download and prepare the dataset to
`DATA_PATH`. Subsequent runs read from the local cache.

The dataset is split 85/10/5 (train/val/test). Samples are preprocessed to extract MFCC features and delivered as uint8 values (0–255).
Training applies data augmentation through time shifting, time masking and frequency masking (see `speech_commands_data_loader.py`).

To pre-download without training, run:
```bash
python -c "import tensorflow_datasets as tfds; tfds.load('speech_commands', data_dir='./data/sc10')"
```

In [ ]:
import yaml

from speech_commands_data_loader import compute_mfcc_range, get_datasets

with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

SEED = cfg['seed']

data_transform = compute_mfcc_range(data_dir=DATA_PATH)
train_ds, test_ds, val_ds = get_datasets(
    data_dir=DATA_PATH,
    batch_size=cfg["batch_size"],
    data_transform=data_transform,
    aug_enabled=cfg.get("aug_enabled", False),
    aug_time_shift_max_ms=cfg.get("aug_time_shift_max_ms", 100),
    aug_freq_mask_param=cfg.get("aug_freq_mask_param", 2),
    aug_time_mask_param=cfg.get("aug_time_mask_param", 10),
    shuffle_seed=SEED,
    aug_seed=SEED)

I0000 00:00:1785958368.905416 1238149 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22284 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:04:00.0, compute capability: 8.6


MFCC range [0.5–99.5 pct]: -51.4311 – 14.2340


## Model

Akida 1.0 (AKD1500) supports a well-defined set of layer operations. The reference DS-CNN
model as defined in MLPerf Tiny requires a small number of targeted changes to run entirely on-chip. Each change
is described below:

**1. Input convolution kernel shape: `(10, 4)` → `(5, 5)`**  
Akida 1.0's input convolution layer requires a **square kernel** of size 1, 3, 5, or 7.
The original `(10, 4)` kernel violates both constraints. A `(5, 5)` kernel is a practical
replacement — it covers a similar receptive field area and works well empirically on this task.

**2. Depthwise + Pointwise → Fused Separable blocks**  
In the original model, a ReLU activation is applied after the depthwise convolution and again
after the pointwise convolution. Akida 1.0 does not support an activation between the depthwise
and pointwise steps of a separable block — the two must be **fused** (a single activation after
the pointwise layer only). The `separable_conv_block` helper from `akida_models` produces this
correct fused structure.

**3. Global Average Pooling position: after ReLU → before ReLU**  
In Akida 1.0, the Global Average Pooling operation must come **before** the final ReLU in the
last separable block, not after it. The `post_relu_gap=False` argument to `separable_conv_block`
places the pooling correctly.

**4. Add a `Rescaling` layer at the input**  
The Akida hardware operates on **uint8 inputs** (values 0–255). Adding a `Rescaling` layer as
the first operation in the model folds the [0, 255] → [0, 1] normalization directly into the
model weights, so no external pre-processing is needed at inference time. The hardware passes
raw uint8 data in and the on-chip rescaling is handled transparently.

**5. Remove the output `softmax`; train with `from_logits=True`**  
Softmax is not implemented as an on-chip operation in Akida 1.0. Removing it and training with
`SparseCategoricalCrossentropy(from_logits=True)` avoids spurious warnings at conversion time.
For inference, classification is simply the `argmax` of the raw output logits — equivalent in
result, since softmax is monotonic and does not change the argmax.

In [3]:
from speech_commands_model import build_ds_cnn

model = build_ds_cnn(
    filters=cfg["filters"],
    dropout_initial=cfg["dropout_initial"],
    dropout_final=cfg["dropout_final"],
    weight_decay=cfg["weight_decay"],
    num_sep_conv_blocks=cfg.get("num_sep_conv_blocks", 4),
    classifier_head=cfg.get("classifier_head", "dense"),
    seed=SEED
)
model.summary()

Model: "ds_cnn"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 49, 10, 1)]       0         
                                                                 
 rescaling (Rescaling)       (None, 49, 10, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 25, 5, 64)         1664      
                                                                 
 conv2d/BN (BatchNormalizat  (None, 25, 5, 64)         256       
 ion)                                                            
                                                                 
 conv2d/relu (ReLU)          (None, 25, 5, 64)         0         
                                                                 
 dropout (Dropout)           (None, 25, 5, 64)         0         
                                                            

## Float Training

The model is trained in full float32 precision for 30 epochs using the Adam
optimiser and sparse categorical cross-entropy loss (with `from_logits=True`,
since the model head outputs raw logits rather than softmax probabilities).

The learning rate follows a cosine decay schedule with linear warmup, starting at `1e-6`, warming up to the configured peak learning rate (`cfg["lr_float"]`) over the first 10% of training steps, then decaying back toward 0 by the final epoch. 

**Increasing Sparsity with Activity Regularization**: ReLU activations already produce some sparsity (any negative pre-activation becomes zero),
but we can encourage further sparsity by adding an **activity regularizer** directly to the
activation layers. This adds a small penalty to the loss function, incentivizing the model to drive more activations toward zero during
training.

We use **Hoyer-square regularization** on the ReLU outputs, which computes the ratio of the L1 and L2 norms.
Its implementation can be found at `regularizers_custom.py`.

Set `RUN_FLOAT_TRAINING = True` above to train from scratch. Otherwise, the
cell below loads a float model from the `pretrained_models` folder.

In [4]:
from speech_commands_train import train_speech_commands

if RUN_FLOAT_TRAINING:
    warmup_fraction = cfg.get("warmup_fraction", 0.1)
    act_reg_strength = cfg["activity_reg_hoyer_strength"]
    epochs = cfg["epochs_float"]
    peak_lr = cfg["lr_float"]
    train_speech_commands(model=model,
                          train_ds=train_ds,
                          val_ds=val_ds,
                          epochs=epochs,
                          peak_lr=peak_lr,
                          warmup_fraction=warmup_fraction,
                          act_reg_strength=act_reg_strength,
                          seed=SEED)
    model.save(
        MODELS_DIR + 'speech_commands.h5',
        include_optimizer=False)
    print('Float model saved.')
else:
    float_model_path = 'pretrained_models/speech_commands.h5'
    model = load_quantized_model(float_model_path)
    model.compile(metrics=['accuracy'])

Adding Activity Regularization (Hoyer-Square) to ReLU layers
Epoch 1/30


E0000 00:00:1785958376.590675 1238149 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inds_cnn/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1785958376.934899 1238242 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1785958378.043226 1238249 service.cc:152] XLA service 0x793e3ed8cdd0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1785958378.043243 1238249 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
I0000 00:00:1785958378.111536 1238249 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1337/1337 [==============================] - 22s 13ms/step - loss: 1.7201 - accuracy: 0.5732 - val_loss: 2.3125 - val_accuracy: 0.6215
Epoch 2/30
1337/1337 [==============================] - 17s 13ms/step - loss: 1.0483 - accuracy: 0.6958 - val_loss: 2.3170 - val_accuracy: 0.6277
Epoch 3/30
1337/1337 [==============================] - 18s 13ms/step - loss: 0.8889 - accuracy: 0.7385 - val_loss: 2.5390 - val_accuracy: 0.6666
Epoch 4/30
1337/1337 [==============================] - 17s 12ms/step - loss: 0.8116 - accuracy: 0.7625 - val_loss: 1.7186 - val_accuracy: 0.7122
Epoch 5/30
1337/1337 [==============================] - 18s 13ms/step - loss: 0.7602 - accuracy: 0.7824 - val_loss: 1.4062 - val_accuracy: 0.7164
Epoch 6/30
1337/1337 [==============================] - 18s 13ms/step - loss: 0.7315 - accuracy: 0.7902 - val_loss: 0.9189 - val_accuracy: 0.7797
Epoch 7/30
1337/1337 [==============================] - 18s 13ms/step - loss: 0.7106 - accuracy: 0.7994 - val_loss: 0.5040 - val_accura

/home/dmclelland/miniconda3/envs/brainchip_devhub_env/lib/python3.12/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [5]:
_, float_acc = model.evaluate(test_ds, verbose=1)
print(f'Float accuracy (test): {float_acc:.4f}')
_, float_acc = model.evaluate(val_ds, verbose=1)
print(f'Float accuracy (val): {float_acc:.4f}')

77/77 [==============================] - 0s 3ms/step - loss: 0.3255 - accuracy: 0.9151
Float accuracy (test): 0.9151
158/158 [==============================] - 1s 3ms/step - loss: 0.2087 - accuracy: 0.9569
Float accuracy (val): 0.9569


## Quantization

Post-training quantization (PTQ) via `cnn2snn.quantize` converts the model
to fixed-point arithmetic:
- **Input**: 8-bit (`-i 8`)
- **Weights**: 4-bit (`-w 4`)
- **Activations**: 4-bit (`-a 4`)

4-bit quantization must be used to be compatible with Akida 1 hardware. Note though that
the first layer (both its inputs and weights) can be 8-bit.

In [6]:
import cnn2snn

quantized_model = cnn2snn.quantize(
    model,
    input_weight_quantization=8,
    weight_quantization=4,
    activ_quantization=4)
print('Model quantized to i8/w4/a4.')

Model quantized to i8/w4/a4.


Quantizing a model after training like this is referred to as Post-Training
Quantization (PTQ). It can slightly reduce accuracy (especially at 4-bits as
here) because the model was trained with continuous weights but is now 
evaluated with discrete values.

In [7]:
quantized_model.compile(metrics=['accuracy'])
_, ptq_acc = quantized_model.evaluate(test_ds, verbose=1)
print(f'PTQ accuracy (test): {ptq_acc:.4f}')
_, ptq_acc = quantized_model.evaluate(val_ds, verbose=1)
print(f'PTQ accuracy (val): {ptq_acc:.4f}')

77/77 [==============================] - 1s 3ms/step - loss: 0.1347 - accuracy: 0.8022
PTQ accuracy (test): 0.8022
158/158 [==============================] - 1s 3ms/step - loss: 0.1347 - accuracy: 0.8908
PTQ accuracy (val): 0.8908


## Quantization-Aware Training (QAT)

We can run Quantization Aware Training (QAT) to recover most of the drop in 
accuracy. QAT fine-tunes the quantized model for a few epochs (here, 25) at a
reduced learning rate. Note that, although it can sound intimidating,
QAT with BrainChip's quantization tools is no more complex than simply sending
the quantized model back through the same training pipeline that was used to
prepare the float model in the first place.

Here too `Hoyer-square` regulrization is applied to the model's activations.

Set `RUN_QAT_TRAINING = True` above to run QAT locally. Otherwise, the
cell below loads a QAT model from the `pretrained_models` folder.

In [ ]:
if RUN_QAT_TRAINING:
    # We refetch the dataset, only to approach reproducibility against the non-notebook pipeline.
    # This resets the shuffle seed on the training data
    # (Even so, QAT likely will not perfectly match the script version: we believe this is an
    # issue with Dropout layer initialisation)
    train_ds, test_ds, val_ds = get_datasets(
        data_dir=DATA_PATH,
        batch_size=cfg["batch_size"],
        data_transform=data_transform,
        aug_enabled=cfg.get("aug_enabled", False),
        aug_time_shift_max_ms=cfg.get("aug_time_shift_max_ms", 100),
        aug_freq_mask_param=cfg.get("aug_freq_mask_param", 2),
        aug_time_mask_param=cfg.get("aug_time_mask_param", 10),
        shuffle_seed=SEED,
        aug_seed=SEED)
    

    epochs = cfg["epochs_qat"]
    peak_lr = cfg["lr_qat"]
    warmup_fraction = cfg.get("warmup_fraction", 0.1)
    act_reg_strength = cfg["activity_reg_hoyer_strength_qat"]
    train_speech_commands(model=quantized_model,
                            train_ds=train_ds,
                            val_ds=val_ds,
                            epochs=epochs,
                            peak_lr=peak_lr,
                            warmup_fraction=warmup_fraction,
                            act_reg_strength=act_reg_strength,
                            seed=SEED)

    quantized_model.save(
        MODELS_DIR + 'speech_commands_qat.h5',
        include_optimizer=False)
    print('QAT model saved.')
else:
    qat_model_path = 'pretrained_models/speech_commands_qat.h5'
    quantized_model = load_quantized_model(qat_model_path)
    quantized_model.compile(metrics=['accuracy'])


Adding Activity Regularization (Hoyer-Square) to ReLU layers
Epoch 1/25


E0000 00:00:1785958923.896778 1238149 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_3/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


1337/1337 [==============================] - 25s 16ms/step - loss: 2.1149 - accuracy: 0.7922 - val_loss: 1.6173 - val_accuracy: 0.9241
Epoch 2/25
1337/1337 [==============================] - 21s 16ms/step - loss: 1.9288 - accuracy: 0.8054 - val_loss: 1.5202 - val_accuracy: 0.9209
Epoch 3/25
1337/1337 [==============================] - 22s 16ms/step - loss: 1.8012 - accuracy: 0.8075 - val_loss: 1.3900 - val_accuracy: 0.9249
Epoch 4/25
1337/1337 [==============================] - 22s 16ms/step - loss: 1.7055 - accuracy: 0.8109 - val_loss: 1.4029 - val_accuracy: 0.9267
Epoch 5/25
1337/1337 [==============================] - 22s 16ms/step - loss: 1.6732 - accuracy: 0.8087 - val_loss: 1.3304 - val_accuracy: 0.9267
Epoch 6/25
1337/1337 [==============================] - 22s 16ms/step - loss: 1.6405 - accuracy: 0.8095 - val_loss: 1.3142 - val_accuracy: 0.9213
Epoch 7/25
1337/1337 [==============================] - 22s 16ms/step - loss: 1.6170 - accuracy: 0.8107 - val_loss: 1.2886 - val_accura

In [9]:
_, qat_acc = quantized_model.evaluate(test_ds, verbose=1)
print(f'QAT accuracy (test): {qat_acc:.4f}')
_, qat_acc = quantized_model.evaluate(val_ds, verbose=1)
print(f'QAT accuracy (val): {qat_acc:.4f}')

77/77 [==============================] - 1s 6ms/step - loss: 1.3607 - accuracy: 0.8679
QAT accuracy (test): 0.8679
158/158 [==============================] - 1s 6ms/step - loss: 1.2057 - accuracy: 0.9287
QAT accuracy (val): 0.9287


## Conversion to Akida Format

`cnn2snn.convert` compiles the quantized Keras model into an Akida `.fbz`
model that can be loaded and executed directly on AKD1500 hardware.
The converter verifies hardware compatibility and maps each layer to its
corresponding Akida primitive.

In [10]:
akida_model = cnn2snn.convert(quantized_model)

akida_model_path = os.path.join(MODELS_DIR, 'speech_commands_qat.fbz')
akida_model.save(akida_model_path)
print(f'Akida model saved to {akida_model_path}')
akida_model.summary()

Akida model saved to ./models_notebook/speech_commands_qat.fbz
                Model Summary                 
______________________________________________
Input shape  Output shape  Sequences  Layers
[49, 10, 1]  [1, 1, 12]    1          6     
______________________________________________

______________________________________________________________
Layer (type)                    Output shape  Kernel shape  

================ SW/conv2d-conv2d_1 (Software) ===============

conv2d (InputConv.)             [25, 5, 64]   (5, 5, 1, 64) 
______________________________________________________________
separable_conv2d (Sep.Conv.)    [25, 5, 64]   (3, 3, 64, 1) 
______________________________________________________________
                                              (1, 1, 64, 64)
______________________________________________________________
separable_conv2d_1 (Sep.Conv.)  [25, 5, 64]   (3, 3, 64, 1) 
______________________________________________________________
                    

/home/dmclelland/miniconda3/envs/brainchip_devhub_env/lib/python3.12/site-packages/cnn2snn/transforms/act_step_equalization.py:89: UserWarning: The conv2d layer holds very high threshold values which are not compatible with the hardware. Those are clipped to the maximum supported value 524287. Continuing execution.
  warnings.warn(f"The {layer.name} layer holds very high threshold values which are "


## Evaluation of Akida Model

We now run evaluation through the Akida model, to check that accuracy is 
comparable to that obtained from the quantized tf_keras model. Here, we deliberately
use the software backend (the default, since we do not check for and map to
a connected hardware device): this delivers a
bit-accurate simulation of the results that will be obtained when running
the model on hardware.

In the accompanying [speech_commands_notebook_benchmark.ipynb](speech_commands_notebook_benchmark.ipynb)
the same evaluation is run using the hardware backend (if, of course, a hardware Akida
device is connected), allowing you to confirm that the results are identical.

### Run Evaluation on Akida

The Akida runtime cannot consume `tf.data.Dataset` objects directly, rather
it expects a 4D numpy array (n, h, w, c) in uint8 format. So we
iterate over validation batches manually.

The model output tensor has shape `(B, 1, 1, C)` which is squeezed to 
`(B, C)` before taking the class argmax.

In [11]:
from tqdm import tqdm

BATCH_SIZE = cfg["batch_size"]

labels_all = []
logits_all = []
for batch, label_batch in tqdm(val_ds, desc="Evaluating on Akida"):
    if not isinstance(batch, np.ndarray):
        batch = batch.numpy()

    logits_batch = akida_model.predict(batch, batch_size=BATCH_SIZE)

    logits_batch = logits_batch.squeeze(axis=(1, 2))
    labels_all.append(label_batch)
    logits_all.append(logits_batch)

labels_all = np.concatenate(labels_all)
logits_all = np.concatenate(logits_all)
preds = np.argmax(logits_all, axis=1)

akida_acc = float(np.mean(preds == np.array(labels_all)))
print(f'Akida accuracy (val): {akida_acc:.4f}')

Evaluating on Akida: 100%|██████████| 158/158 [00:01<00:00, 93.13it/s] 

Akida accuracy (val): 0.9286


### Activation Sparsity

Akida hardware skips computation for zero-valued activations, so activation
sparsity directly reduces both energy consumption and inference latency.
Below we measure per-layer sparsity on a 1024-sample calibration batch drawn
from the validation set.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from speech_commands_data_loader import get_samples

NUM_SAMPLES = 1024

samples = get_samples(DATA_PATH, data_transform=data_transform, num_samples=NUM_SAMPLES)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)


Layer                  Sparsity
-------------------------------
conv2d                  92.72%
separable_conv2d        92.46%
separable_conv2d_1      96.24%
separable_conv2d_2      98.30%
separable_conv2d_3      59.00%
conv2d_1                 0.00%
-------------------------------
Mean                    73.12%


## Summary

The table below compares validation accuracy across the three model variants.
The goal is that QAT and Akida accuracy remain close to the float baseline.

In [13]:
print('Speech Commands results')
print('=' * 40)
print(f'  Float accuracy:     {float_acc * 100:.2f}%')
print(f'  QAT accuracy:       {qat_acc * 100:.2f}%')
print(f'  Akida accuracy:     {akida_acc * 100:.2f}%')

Speech Commands results
  Float accuracy:     95.69%
  QAT accuracy:       92.87%
  Akida accuracy:     92.86%
